
---
  Q4: Miss Rate with Code [2024 Q2b(iv)]

  Direct-mapped cache, cold, cache-aligned. Block size 16 bytes, cache 64 bytes (4 sets). Float =
   8 bytes.

  float num[16], result = 0;
  for (int i = 0; i < 8; i++)
      for (int j = 0; j < 2; j++)
          result += num[j*8 + i];

  Access order: num[0], num[8], num[1], num[9], num[2], num[10]...

  What's the miss rate? (Trace which set each access maps to and whether the previous occupant
  was evicted.)

---
   This is a beautiful, evil, classic exam question. It is designed to test if
   you understand the absolute worst-case scenario for a direct-mapped cache:
   THRASHING.

   When you write C code, you might think, "I'm accessing my array in a 
   predcitable pattern, so the cache will save me!" This question proves how a
   tiny loop swap can destroy your performance, resulting in a 100% MISS RATE.


STEP 1: The Geometry of the Cache (The Parking Lot)
   Before we look at the code, we have to figure out how big our "parking space"
   are and how many we have.

   - The Data: A `float` in C is 8 bytes.
   - The Block (The Car): The cache block size is 16 bytes. This means exactly
     TWO `float`s fit into one block. When the `CPU` asks for `num[0]`, the 
     cache will automatically grab `num[0]` and `num[1]` together and park them
     as a single unit.
   - The Sets (The Parking Spots): The total cache is 64 bytes. Since each block
     is 16 bytes, the cache has room for exactly 4 BLOCKS (cache size / block size = block number)
     . Because it's "direct-mapped," each set holds 1 block. So, we have 4 Sets
     (Set 0, 1, 2, and 3).
     

STEP 2: The Mapping (Who Parks Where?)
   The prompt says the array is "cache-aligned." This means `num[0]` perfectly 
   starts at the beginning of Set 0.

   Because we only have 4 sets, the cache acts like a loop. It puts Block 0 in 
   Set 0, Block 1 in Set 1, Block 2 in Set 2... and then it wraps around!
   Block 4 is forced to park in Set 0.

   Let's map out the first few elements:
      Block 0 (num[0], num[1]) $\rightarrow$ Set 0
      Block 1 (num[2], num[3]) $\rightarrow$ Set 1
      Block 2 (num[4], num[5]) $\rightarrow$ Set 2
      Block 3 (num[6], num[7]) $\rightarrow$ Set 3
      Block 4 (num[8], num[9]) $\rightarrow$ Set 0 (Uh oh. Notice the collision!)


STEP 3: The Code's Sabotage
   Look closely at the nested loops. The inner loop is `j` (which flips between
   0 and 1), and the outer loop is `i`.

   The formula is `num[j*8 + i]`.

   Let's trace the first 4 memory accesses ...
        i=0, j=0: num[0]
        i=0, j=1: num[8]
        i=1, j=0: num[1]
        i=1, j=1: num[9]


STEP 4: The Execution (The Thrashing)
   Now's let's play the role of the cache hardware and execute those first four
   accesses.

   1. CPU asks for `num[0]`: The cache is cold (empty). MISS. The cache goes to
      RAM, fetches Block 0 (`num[0]` and `num[1]`), and parks it in SET 0.
   2. CPU asks for `num[8]`: The cache looks in SET 0 (where `num[8]` is 
      supposed to live). But Block 0 is parked there! MISS. The cache kicks
      out Block 0, goes to RAM, fetches Block 4 (`num[8]` and `num[9]`), and 
      parks it in Set 0.
   3. ... MISS... kick... park
   4. ...

   This is called CONFLICT MISS THRASHING.

   Even though the cache is easily big enough to hold both of these blocks at
   the same time, the hardware rules of a Direct-Mapped cache force them to
   fight to the death over SET 0. Because the C code alternates perfectly 
   between them, the cache evicts the exact data it is about to need on the
   very next line of code. 

   This pattern reepats for the entire loop. Every single access evicts the
   previouis one... RESULT: 16 accesses, 16 misses. 100% Miss Rate





---
  Q5: Matrix Multiply Miss Rate [Tutorial 40005_7]

  // Standard: C = A*B -- inner loop accesses A[i][k] and B[k][j]
  // Transposed: C = A*B^T -- inner loop accesses A[i][k] and B[j][k]

  Cache: 8 MiB, 16-way, 64-byte blocks. N is huge. float=4 bytes. C[i][j] in register.

  (a) Miss rate for standard multiply?
  (b) Miss rate for transposed multiply?

  Hint: In the inner loop over k, A[i][k] is stride-1 in both versions. B[k][j] is stride-N
  (column access = bad), but B[j][k] is stride-1 (row access = good). 64 bytes / 4 bytes = 16
  floats per block, so stride-1 misses 1/16 of the time.